```{tip}
**For the presenter:** before starting the slideshow, run *Kernel → Restart Kernel and Run All Cells*, so all widgets are shown.
```

# 02.1 First Model

## Contents

**Tools:** CADET-GUI widgets: components, units, flow sheet, process and simulation; CADET-Process (Python).

**Python used:** Imports, object creation, attribute access and method calls.

**Introduces:** `ComponentSystem`, `Inlet`, `Outlet`, `LumpedRateModelWithoutPores`, `FlowSheet`, `Process`, tracer breakthrough and liquid/bound states.

1. The experiment
2. Components
3. Unit operations: inlet, outlet, column
4. Flow sheet
5. Process
6. Simulation: when does the tracer arrive?

```{note}
The model is built live in this lesson; there is no need to type along. The complete code is on the `solution` branch, and the **02.1 First Model: Exercises** build the model step by step.
```

```{dropdown} Working with Jupyter notebooks
- **Shift+Enter** runs a cell and moves on; **Ctrl+Enter** runs it and stays.
- Run cells **from top to bottom**: later cells use what earlier cells created.
- `[3]`: third cell run; `[*]`: still running.
- Start over: *Kernel → Restart Kernel and Run All Cells*.
- Collapsed cells: click the bar on their left.
```

```{dropdown} Python in a nutshell
- **Import** a class: `from CADETProcess.processModel import Inlet`
- **Create** an object: `inlet = Inlet(component_system, "inlet")`
- **Attributes** hold values: `inlet.flow_rate = 1e-7`
- **Call a method** with parentheses: `flow_sheet.add_unit(inlet)` adds the inlet to the flow sheet.
- **Access nested objects** with dots: `process.flow_sheet.column.length` reads the length of the column in the process's flow sheet.
```

In [ ]:
# Load the shared GUI setup functions.
%run ../resources/first_steps/lesson_setup.py
# Create empty model objects for the widgets below to fill.
component_system, units, flow_sheet, process = empty_model("tracer")

## The experiment

A typical protein purification starts with **loading** a mixture, **washing** away unbound material and **eluting** the retained proteins with a salt gradient. A model helps choose the flow rate and gradient before trying them in the lab (03.1 Simple Chromatography).

We build towards that workflow: first characterize transport with a tracer, then add inlet profiles, instrument delay and binding. You have a freshly packed column. Before running the experiment, you would like to know:

1. When does the tracer reach the outlet? (this lesson: Simulation)
2. What changes at another flow rate or on a bigger column? (02.1 First Model: exercises 1 and 2)
3. How much does transport spread out the tracer's arrival times? (02.4 Residence Time Distributions)

The standard first experiment: pump a **tracer**, a molecule that does not bind (e.g. acetone or salt), through the column from $t = 0$.
The detector at the outlet records the **breakthrough curve**: the concentration rises from zero to the feed concentration.

### The column and the feed

| Parameter          | Symbol          | Value | Unit                           |
| ------------------ | --------------- | ----- | ------------------------------ |
| Column length      | $L$             | 10    | $\text{cm}$                    |
| Column diameter    | $d$             | 2     | $\text{cm}$                    |
| Total porosity     | $\varepsilon_t$ | 0.4   | \-                             |
| Axial dispersion   | $D_\text{ax}$   | 1e-7  | $\text{m}^2 \text{s}^{-1}$     |
| Flow rate          | $Q$             | 6     | $\text{mL} \, \text{min}^{-1}$ |
| Feed concentration | $c_\text{feed}$ | 1     | $\text{mol} \, \text{m}^{-3}$  |

Total porosity and axial dispersion are estimates (typical values). We estimate porosity from a measurement in the 02.4 Residence Time Distributions exercises.

```{note}
CADET does not convert units: all values must be given in one consistent system.
We use SI units throughout, e.g. $Q = 6~\text{mL}\,\text{min}^{-1} = 10^{-7}~\text{m}^3\,\text{s}^{-1}$.
```

## Components

The **component system** defines the species; all other objects refer to it.
We enter `Tracer` in **Components**. This defines the single species used by every unit. **Show code** reveals its Python representation.

In [ ]:
# Add the tracer component on the solution branch.
component_system.add_component("Tracer")

In [ ]:
from cadetgui import ComponentSystemWidget

ComponentSystemWidget(component_system).display()

## Unit operations

```{figure} ../resources/first_steps/unit_operation_examples.svg
:width: 100%
:align: center
:alt: Unit operation examples: inlet, CSTR, tubing, column and outlet, each with its CADET-Process class.
```

One piece of apparatus and the model of its physics: e.g. a column, a mixing tank or a piece of tubing. Inlets and outlets are unit operations, too: they supply and collect the liquid. Each has its own CADET-Process class; this lesson uses an inlet, a column and an outlet.

### Inlet

```{figure} ../resources/first_steps/inlet_parameters.svg
:width: 90%
```

- `c`: concentration of each component
- `flow_rate`: volumetric flow rate $Q$
- Here both are constant: $c = 1~\text{mol}\,\text{m}^{-3}$, $Q = 10^{-7}~\text{m}^3\,\text{s}^{-1}$.

```{dropdown} Profiles that change over time
`c` and `flow_rate` are cubic polynomials in time, $c_\text{in}(t) = a_0 + a_1 t + a_2 t^2 + a_3 t^3$.
A constant feed only needs $a_0$; $a_1$ gives a linear gradient.
More in the lesson on inlet profiles.
```

### Outlet

The sink of the system, without parameters. Here it is our detector.

### Column

```{figure} ../resources/first_steps/column_parameters.svg
:width: 90%
```

- Column volume $V = A L$ with cross-section $A = \pi (d/2)^2$
- Liquid volume $\varepsilon_t V$: the tracer only moves through the liquid.
- **Mean liquid velocity** $u = \dfrac{Q}{\varepsilon_t A}$

### Lumped rate model without pores (LRM)

All liquid, between and inside the particles, is lumped into $\varepsilon_t$. The model state $c(z,t)$ is the **liquid concentration** along the column. For a tracer:

$$
\underbrace{\frac{\partial c}{\partial t}}_{\text{accumulation}} = \underbrace{- u \frac{\partial c}{\partial z}}_{\text{convection}} + \underbrace{D_\text{ax} \frac{\partial^2 c}{\partial z^2}}_{\text{axial dispersion}}
$$

**Model parameters** appear in the equation above:

| Attribute          | Symbol          | Unit                        | Role                                  |
| ------------------ | --------------- | --------------------------- | ------------------------------------- |
| `length`           | $L$             | $\text{m}$                  | length of the column                  |
| `diameter`         | $d$             | $\text{m}$                  | cross-section, so $u$                 |
| `total_porosity`   | $\varepsilon_t$ | \-                          | liquid fraction, so $u$               |
| `axial_dispersion` | $D_\text{ax}$   | $\text{m}^2\,\text{s}^{-1}$ | mixing along the column               |

**Initial conditions** set the state at $t = 0$, from which the equation starts:

| Attribute | Symbol | Unit                        | Meaning                                  |
| --------- | ------ | --------------------------- | ---------------------------------------- |
| `c`       | $c$    | $\text{mol}\,\text{m}^{-3}$ | initial liquid concentration (default 0) |
| `q`       | $q$    | $\text{mol}\,\text{m}^{-3}$ | initial bound concentration (default 0)  |

The bound concentration $q$ does not appear in the tracer equation: without binding, it stays at its initial value of zero. The detector curve is the liquid concentration at the column exit, $c(L,t)$; in Python it is available as `results.solution.column.outlet`. Binding (02.5) introduces an equation for $q$; pore transport (03.1) introduces a separate liquid state inside the particles.

The flow rate is not a column parameter: the column gets it from the units that feed it.
See the [LRM documentation](https://cadet.github.io/master/modelling/unit_operations/lumped_rate_model_without_pores.html).

### Creating the unit operations

For each unit, we select its **Model**, enter the **Name** from the table and click **Create**. These units represent the feed, packed column and detector:

| Model                                 | Name     |
| ------------------------------------- | -------- |
| Inlet                                 | `inlet`  |
| Lumped Rate Model Without Pores (LRM) | `column` |
| Outlet                                | `outlet` |

In [ ]:
# Create the three units without experiment settings on solution.
create_tracer_units(component_system, units)

In [ ]:
# Show only unit-creation code; parameter assignments come next.
creation_widget = unit_creation_widget(component_system, units)
creation_widget.display()

### Setting the parameters

Under **Parameters of**, we select `inlet`, then `column`, and enter the values below in SI units. The column stays without binding; `outlet` needs no settings:

| Unit     | Parameters                                                                 |
| -------- | -------------------------------------------------------------------------- |
| `inlet`  | `c`: $a_0 = 1~\text{mol}\,\text{m}^{-3}$<br>`flow_rate`: $10^{-7}~\text{m}^3\,\text{s}^{-1}$ |
| `column` | `length`: $0.1~\text{m}$<br>`diameter`: $0.02~\text{m}$<br>`total_porosity`: $0.4$<br>`axial_dispersion`: $10^{-7}~\text{m}^2\,\text{s}^{-1}$ |

Parameters belong to the unit operations. See the [unit operation documentation](https://cadet-process.readthedocs.io/en/latest/user_guide/process_model/unit_operation.html).

In [ ]:
# Apply the feed and column parameters at this step on solution.
set_tracer_parameters(units)

In [ ]:
compact_unit_widget(component_system, units, create=False).display()

## Flow sheet

```{figure} ../resources/first_steps/batch_elution.svg
:width: 500px
:align: center
```

How the unit operations are connected, here for a batch elution.
See the [flow sheet documentation](https://cadet-process.readthedocs.io/en/latest/user_guide/process_model/flow_sheet.html).

### Building the flow sheet

Each unit is added under **Unit to add** with **Add to flow sheet**. With **From** / **To** and **Connect**, we then connect `inlet` → `column` and `column` → `outlet`. The diagram now shows the complete flow path.

In [ ]:
# Add and connect the three units at this step on solution.
connect_tracer_units(units, flow_sheet)

In [ ]:
from cadetgui import FlowSheetWidget

FlowSheetWidget(flow_sheet, units=units).display()

## Process

```{figure} ../resources/first_steps/batch_elution_process_model.svg
:width: 800px
:align: center
```

A process combines the flow sheet with its operation over time. It:

- sets the **cycle time** and the **events** (none here)
- reaches every unit parameter under a **path**, e.g. `flow_sheet.column.length`, as used by events and optimization later

### Creating the process

We set `cycle_time` to 600 s and leave the events empty: the tracer feed is constant. **Parameters by path** shows how the process reaches each unit setting.

In [ ]:
# Set the experiment duration at this step on solution.
process.cycle_time = 600

In [ ]:
from cadetgui import ProcessWidget

ProcessWidget(process).display()

## Simulation: when does the tracer arrive?

Before we simulate: when do you expect the tracer at the outlet?

The tracer does not bind, so it travels with the liquid.
The column holds the liquid volume $\varepsilon_t V$, and the pump replaces it at the flow rate $Q$.
The **mean retention time** follows from liquid volume divided by flow rate:

$$
t_R = \frac{\varepsilon_t V}{Q} = \frac{L}{u}
$$

In [ ]:
import numpy as np

volume = np.pi * (0.02 / 2)**2 * 0.1
t_breakthrough = 0.4 * volume / 1e-7
print(f"t_R = {t_breakthrough:.0f} s = {t_breakthrough / 60:.1f} min")

$t_R$ is the first check of any model: if the simulated front does not arrive around $t_R$, a volume, flow rate or porosity is wrong.
A measured delay can also give column porosity, after accounting for system delay (02.4 Residence Time Distributions exercises).

### Comparing the simulation

```{note}
To simulate a `Process`, we create a `Cadet` simulator object from `CADETProcess.simulator`: `simulator = Cadet()`. Then `simulation_results = simulator.simulate(process)` computes the concentration profiles. The widget below performs these steps on **Run simulation**.
```

After **Run simulation**, selecting `outlet` under **Show** displays the breakthrough; selecting `inlet` shows the constant feed for comparison. The dashed line marks $t_R$.

In [ ]:
from cadetgui import SimulationWidget

SimulationWidget(process, markers={"t_R": t_breakthrough}).display()

Without dispersion, the outlet would jump from zero to the feed concentration exactly at $t_R$.
Axial dispersion spreads the front out around $t_R$: here, it reaches half of the feed concentration at about 126 s, matching our prediction. A broader front means the tracer arrives over a wider range of times.

## Summary

| Level           | Object                        | Parameters                                                   |
| --------------- | ----------------------------- | ------------------------------------------------------------ |
| Components      | `ComponentSystem`             | the species                                                  |
| Unit operations | `Inlet`                       | `c`, `flow_rate`: cubic polynomials in time                  |
|                 | `LumpedRateModelWithoutPores` | `length`, `diameter`, `total_porosity`, `axial_dispersion`   |
|                 | `Outlet`                      | none                                                         |
| Flow sheet      | `FlowSheet`                   | units and connections                                        |
| Process         | `Process`                     | `cycle_time`, events; unit parameters by path                |

**Now:** **02.1 First Model: Exercises**: scale up the column, compare flow rate and porosity, and connect two columns.

**Next lesson:** **02.2 Model Templates** sets up standard experiments on a complete chromatography instrument.